## Merging the Amazon Purchase Data and Survey Data and Dataset Cleaning

In [109]:
%pip install pandas scikit-learn matplotlib seaborn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Merging the Data

In [110]:
# Creating the amazon purchase dataframe as before

import pandas as pd

amazon1 = pd.read_csv('/workspaces/group-project-bas-team/data/AmazonData1.csv')
amazon2 = pd.read_csv('/workspaces/group-project-bas-team/data/AmazonData2.csv')

amazon_data = [amazon1, amazon2]

amazon = pd.concat(amazon_data)

In [111]:
# reading in the survey data

survey = pd.read_csv('/workspaces/group-project-bas-team/data/survey.csv')


In [112]:
# sampling our dataset, since our data contains over 1M rows
# we will sample the survey data, then merge the survey and purchase data with an inner join
# the result will be a dataframe with the complete purchase histories of 1000 customers

survey = survey.sample(1000, random_state=42)

In [113]:
# Merging the Amazon purchase data and survey data on the "Survey ResponseID" column

data = pd.merge(amazon, survey, on='Survey ResponseID', how='inner')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 211405 entries, 0 to 211404
Data columns (total 30 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Order Date                  211405 non-null  str    
 1   Purchase Price Per Unit     211405 non-null  float64
 2   Quantity                    211405 non-null  int64  
 3   Shipping Address State      202738 non-null  str    
 4   Title                       200801 non-null  str    
 5   ASIN/ISBN (Product Code)    211395 non-null  str    
 6   Category                    201119 non-null  str    
 7   Survey ResponseID           211405 non-null  str    
 8   Q-demos-age                 211405 non-null  str    
 9   Q-demos-hispanic            211405 non-null  str    
 10  Q-demos-race                211405 non-null  str    
 11  Q-demos-education           211405 non-null  str    
 12  Q-demos-income              211405 non-null  str    
 13  Q-demos-gender           

### Data Cleaning

In [114]:
# Attempting to identify Title and Category information from matching ASIN/ISBN codes
data[data['Title'].isna()]

,Order Date,Purchase Price Per Unit,Quantity,Shipping Address State,Title,ASIN/ISBN (Product Code),Category,Survey ResponseID,Q-demos-age,Q-demos-hispanic,...,Q-substance-use-marijuana,Q-substance-use-alcohol,Q-personal-diabetes,Q-personal-wheelchair,Q-life-changes,Q-sell-YOUR-data,Q-sell-consumer-data,Q-small-biz-use,Q-census-use,Q-research-society
2,12/24/2018,8.99,1,NJ,NaN,B078JZTFN3,NaN,R_01vNIayewjIIKMF,35 - 44 years,Yes,...,No,No,No,No,NaN,Yes if I get part of the profit,Yes if consumers get part of the profit,No,No,Yes
9,4/23/2019,24.69,1,NJ,NaN,B06XKNWJN2,NaN,R_01vNIayewjIIKMF,35 - 44 years,Yes,...,No,No,No,No,NaN,Yes if I get part of the profit,Yes if consumers get part of the profit,No,No,Yes
36,10/7/2019,11.94,1,NJ,NaN,B07CZ6JCZS,NaN,R_01vNIayewjIIKMF,35 - 44 years,Yes,...,No,No,No,No,NaN,Yes if I get part of the profit,Yes if consumers get part of the profit,No,No,Yes
41,10/18/2019,14.26,1,NJ,NaN,B00KVM2SSO,NaN,R_01vNIayewjIIKMF,35 - 44 years,Yes,...,No,No,No,No,NaN,Yes if I get part of the profit,Yes if consumers get part of the profit,No,No,Yes
59,12/22/2019,14.33,1,NJ,NaN,B00KVM2SSO,NaN,R_01vNIayewjIIKMF,35 - 44 years,Yes,...,No,No,No,No,NaN,Yes if I get part of the profit,Yes if consumers get part of the profit,No,No,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211073,5/2/2019,16.99,1,PA,NaN,B07N317MXV,NaN,R_3GoVgnf4Ih55nVX,25 - 34 years,No,...,Yes,No,No,No,Moved place of residence,Yes if I get part of the profit,Yes if consumers get part of the profit,I don't know,I don't know,Yes
211104,10/18/2019,109.99,1,PA,NaN,B07MDK1GMJ,NaN,R_3GoVgnf4Ih55nVX,25 - 34 years,No,...,Yes,No,No,No,Moved place of residence,Yes if I get part of the profit,Yes if consumers get part of the profit,I don't know,I don't know,Yes
211117,12/28/2019,15.99,1,PA,NaN,B076LF8N8X,NaN,R_3GoVgnf4Ih55nVX,25 - 34 years,No,...,Yes,No,No,No,Moved place of residence,Yes if I get part of the profit,Yes if consumers get part of the profit,I don't know,I don't know,Yes
211125,1/30/2020,23.69,1,DE,NaN,B07WV9VK71,NaN,R_3GoVgnf4Ih55nVX,25 - 34 years,No,...,Yes,No,No,No,Moved place of residence,Yes if I get part of the profit,Yes if consumers get part of the profit,I don't know,I don't know,Yes


In [115]:
codes = data[['ASIN/ISBN (Product Code)', 'Title']].drop_duplicates().dropna()

In [116]:
codes_dict = codes.set_index('ASIN/ISBN (Product Code)')['Title'].to_dict()


In [117]:
# Merging the new codes dataframe with our data to fill in product titles
data['Title'] = data['ASIN/ISBN (Product Code)'].map(codes_dict)


data.info()

<class 'pandas.DataFrame'>
RangeIndex: 211405 entries, 0 to 211404
Data columns (total 30 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Order Date                  211405 non-null  str    
 1   Purchase Price Per Unit     211405 non-null  float64
 2   Quantity                    211405 non-null  int64  
 3   Shipping Address State      202738 non-null  str    
 4   Title                       200943 non-null  str    
 5   ASIN/ISBN (Product Code)    211395 non-null  str    
 6   Category                    201119 non-null  str    
 7   Survey ResponseID           211405 non-null  str    
 8   Q-demos-age                 211405 non-null  str    
 9   Q-demos-hispanic            211405 non-null  str    
 10  Q-demos-race                211405 non-null  str    
 11  Q-demos-education           211405 non-null  str    
 12  Q-demos-income              211405 non-null  str    
 13  Q-demos-gender           

In [118]:
# Repeating this process to fill in categories

cats = data[['ASIN/ISBN (Product Code)', 'Category']].drop_duplicates().dropna()
cats_dict = cats.set_index('ASIN/ISBN (Product Code)')['Category'].to_dict()

In [119]:
data['Category'] = data['ASIN/ISBN (Product Code)'].map(cats_dict)


data.info()

<class 'pandas.DataFrame'>
RangeIndex: 211405 entries, 0 to 211404
Data columns (total 30 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Order Date                  211405 non-null  str    
 1   Purchase Price Per Unit     211405 non-null  float64
 2   Quantity                    211405 non-null  int64  
 3   Shipping Address State      202738 non-null  str    
 4   Title                       200943 non-null  str    
 5   ASIN/ISBN (Product Code)    211395 non-null  str    
 6   Category                    201203 non-null  str    
 7   Survey ResponseID           211405 non-null  str    
 8   Q-demos-age                 211405 non-null  str    
 9   Q-demos-hispanic            211405 non-null  str    
 10  Q-demos-race                211405 non-null  str    
 11  Q-demos-education           211405 non-null  str    
 12  Q-demos-income              211405 non-null  str    
 13  Q-demos-gender           

In [120]:
# Replacing missing values in Q-life_changes "none" to indicate no life changes selected

data['Q-life-changes'] = data['Q-life-changes'].fillna('None')


In [121]:
data.isnull().sum()

Order Date                        0
Purchase Price Per Unit           0
Quantity                          0
Shipping Address State         8667
Title                         10462
ASIN/ISBN (Product Code)         10
Category                      10202
Survey ResponseID                 0
Q-demos-age                       0
Q-demos-hispanic                  0
Q-demos-race                      0
Q-demos-education                 0
Q-demos-income                    0
Q-demos-gender                    0
Q-sexual-orientation              0
Q-demos-state                     0
Q-amazon-use-howmany              0
Q-amazon-use-hh-size              0
Q-amazon-use-how-oft              0
Q-substance-use-cigarettes        0
Q-substance-use-marijuana         0
Q-substance-use-alcohol           0
Q-personal-diabetes               0
Q-personal-wheelchair             0
Q-life-changes                    0
Q-sell-YOUR-data                  0
Q-sell-consumer-data              0
Q-small-biz-use             

In [122]:
# Removing the remaining rows where missing values are present
data = data.dropna()

In [123]:
# data.isnull().sum()

In [124]:
data.columns

Index(['Order Date', 'Purchase Price Per Unit', 'Quantity',
       'Shipping Address State', 'Title', 'ASIN/ISBN (Product Code)',
       'Category', 'Survey ResponseID', 'Q-demos-age', 'Q-demos-hispanic',
       'Q-demos-race', 'Q-demos-education', 'Q-demos-income', 'Q-demos-gender',
       'Q-sexual-orientation', 'Q-demos-state', 'Q-amazon-use-howmany',
       'Q-amazon-use-hh-size', 'Q-amazon-use-how-oft',
       'Q-substance-use-cigarettes', 'Q-substance-use-marijuana',
       'Q-substance-use-alcohol', 'Q-personal-diabetes',
       'Q-personal-wheelchair', 'Q-life-changes', 'Q-sell-YOUR-data',
       'Q-sell-consumer-data', 'Q-small-biz-use', 'Q-census-use',
       'Q-research-society'],
      dtype='str')

In [125]:
# removing redundant information from column names
# removing the "Q-" first, then removing the remaining redundant words
data.columns = data.columns.str.replace('Q-', '')\
    .str.replace('demos-', '')\
    .str.replace('amazon-use-', '')\
    .str.replace('substance-use-', '')\
    .str.replace('personal-', '')
data.columns

Index(['Order Date', 'Purchase Price Per Unit', 'Quantity',
       'Shipping Address State', 'Title', 'ASIN/ISBN (Product Code)',
       'Category', 'Survey ResponseID', 'age', 'hispanic', 'race', 'education',
       'income', 'gender', 'sexual-orientation', 'state', 'howmany', 'hh-size',
       'how-oft', 'cigarettes', 'marijuana', 'alcohol', 'diabetes',
       'wheelchair', 'life-changes', 'sell-YOUR-data', 'sell-consumer-data',
       'small-biz-use', 'census-use', 'research-society'],
      dtype='str')

In [ ]:
data.info()

<class 'pandas.DataFrame'>
Index: 192556 entries, 0 to 211404
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Order Date                192556 non-null  str    
 1   Purchase Price Per Unit   192556 non-null  float64
 2   Quantity                  192556 non-null  int64  
 3   Shipping Address State    192556 non-null  str    
 4   Title                     192556 non-null  str    
 5   ASIN/ISBN (Product Code)  192556 non-null  str    
 6   Category                  192556 non-null  str    
 7   Survey ResponseID         192556 non-null  str    
 8   age                       192556 non-null  str    
 9   hispanic                  192556 non-null  str    
 10  race                      192556 non-null  str    
 11  education                 192556 non-null  str    
 12  income                    192556 non-null  str    
 13  gender                    192556 non-null  str    
 14  sexu

: 

In [46]:
# Exploding columns with multiple values ('life-changes' and 'race')

data['race'] = data['race'].str.split(',')
data['life-changes'] = data['life-changes'].str.split(',')

In [47]:
data = data.explode('race')
data = data.explode('life-changes')

In [48]:
data.info()

<class 'pandas.DataFrame'>
Index: 207861 entries, 0 to 198432
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Order Date                207861 non-null  str    
 1   Purchase Price Per Unit   207861 non-null  float64
 2   Quantity                  207861 non-null  int64  
 3   Shipping Address State    207861 non-null  str    
 4   Title                     207861 non-null  str    
 5   ASIN/ISBN (Product Code)  207861 non-null  str    
 6   Category                  207861 non-null  str    
 7   Survey ResponseID         207861 non-null  str    
 8   age                       207861 non-null  str    
 9   hispanic                  207861 non-null  str    
 10  race                      207861 non-null  str    
 11  education                 207861 non-null  str    
 12  income                    207861 non-null  str    
 13  gender                    207861 non-null  str    
 14  sexu

In [49]:
# separating dates into month, day, and year columns
data[['order_month', 'order_day', 'order_year']] = data['Order Date'].str.split('/', expand=True)

data = data.drop('Order Date', axis=1)

data[['order_month', 'order_day', 'order_year']] = data[['order_month', 'order_day', 'order_year']].apply(pd.to_numeric)

data.info()


<class 'pandas.DataFrame'>
Index: 207861 entries, 0 to 198432
Data columns (total 32 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Purchase Price Per Unit   207861 non-null  float64
 1   Quantity                  207861 non-null  int64  
 2   Shipping Address State    207861 non-null  str    
 3   Title                     207861 non-null  str    
 4   ASIN/ISBN (Product Code)  207861 non-null  str    
 5   Category                  207861 non-null  str    
 6   Survey ResponseID         207861 non-null  str    
 7   age                       207861 non-null  str    
 8   hispanic                  207861 non-null  str    
 9   race                      207861 non-null  str    
 10  education                 207861 non-null  str    
 11  income                    207861 non-null  str    
 12  gender                    207861 non-null  str    
 13  sexual-orientation        207861 non-null  str    
 14  stat

In [50]:
# Encoding Categorical Data
# Dropping features not relevant to predicting purchase behavior

cols_to_drop = [
    'Survey ResponseID',
    'sell-YOUR-data',
    'sell-consumer-data',
    'small-biz-use',
    'census-use',
    'research-society'
]

data = data.drop(columns=cols_to_drop)



In [51]:
# Ordinal encoding for columns where the categories have an order
ordinal_mappings = {
    'age': {
        '18 - 24 years': 1,
        '25 - 34 years': 2,
        '35 - 44 years': 3,
        '45 - 54 years': 4,
        '55 - 64 years': 5,
        '65 and older': 6
    },
    'education': {
        'Prefer not to say': 0,
        'Some high school or less': 1,
        'High school diploma or GED': 2,
        "Bachelor's degree": 3,
        'Graduate or professional degree (MA, MS, MBA, PhD, JD, MD, DDS, etc)': 4
    },
    'income': {
        'Prefer not to say': 0,
        'Less than $25,000': 1,
        '$25,000 - $49,999': 2,
        '$50,000 - $74,999': 3,
        '$75,000 - $99,999': 4,
        '$100,000 - $149,999': 5,
        '$150,000 or more': 6
    },
    'howmany': {
        '1 (just me!)': 1,
        '2': 2,
        '3': 3,
        '4+': 4
    },
    'hh-size': {
        '1 (just me!)': 1,
        '2': 2,
        '3': 3,
        '4+': 4
    },
    'how-oft': {
        'Less than 5 times per month': 1,
        '5 - 10 times per month': 2,
        'More than 10 times per month': 3
    }
}

for col, mapping in ordinal_mappings.items():
    data[col] = data[col].map(mapping)
    unmapped = data[col].isna().sum()
    if unmapped > 0:
        print(f"Warning: {unmapped} unmapped values in '{col}' — check category strings match exactly")

print("Ordinal encoding complete.")
data[list(ordinal_mappings.keys())].head()

Ordinal encoding complete.


,age,education,income,howmany,hh-size,how-oft
0,5,3,2,1,2,1
1,5,3,2,1,2,1
2,5,3,2,1,2,1
3,5,3,2,1,2,1
4,5,3,2,1,2,1


In [52]:
# Binary encoding
# hispanic is our only true binary column

yes_no_map = {'Yes': 1, 'No': 0}

data['hispanic'] = data['hispanic'].map(yes_no_map)

In [53]:
# One hot encoding
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

data_one_hot = encoder.fit_transform(data[['cigarettes', 'alcohol', 'marijuana', 'diabetes',\
                                            'wheelchair', 'Shipping Address State', 'gender',\
                                            'sexual-orientation', 'state', 'race', 'life-changes']])
df_one_hot = pd.DataFrame(data_one_hot, columns=encoder\
                          .get_feature_names_out(['cigarettes', 'alcohol', 'marijuana', 'diabetes',\
                                                    'wheelchair', 'Shipping Address State', 'gender',\
                                                    'sexual-orientation', 'state', 'race', 'life-changes']))

In [54]:
# Remove one column from each variable to prevent multicollinearity
df_one_hot = df_one_hot.drop(columns=['cigarettes_Prefer not to say', 'alcohol_Prefer not to say', \
                                      'marijuana_Prefer not to say', 'diabetes_Prefer not to say', \
                                        'wheelchair_Prefer not to say', 'Shipping Address State_I did not reside in the United States', \
                                        'gender_Prefer not to say', 'sexual-orientation_Prefer not to say', 'state'])

KeyError: "['wheelchair_Prefer not to say', 'Shipping Address State_I did not reside in the United States', 'gender_Prefer not to say', 'sexual-orientation_Prefer not to say', 'state'] not found in axis"

In [ ]:
# Add df_one_hot to existing data and remove original columns

In [ ]:
# Still need to 

In [ ]:
# No need to encode Title, ASIN/ISBN (Product Code), and category yet, since these will be our potential target values

In [ ]:
# Binary encoding for Yes/No columns

# binary_cols = [
#     'hispanic',
#     'cigarettes',
#     'alcohol',
#     'diabetes',
#     'wheelchair'
# ]

# Preview unique values first to confirm mapping
# for col in binary_cols:
#     print(f"{col}: {data[col].unique()}")

In [ ]:
# Mapping Yes/No to 1/0

# yes_no_map = {'Yes': 1, 'No': 0}

# for col in ['hispanic', 'diabetes', 'wheelchair']:
#     data[col] = data[col].map(yes_no_map)

# Substance use columns have more nuanced values — encode as current user (1) or not (0)
# substance_map = {
#     'Yes': 1,
#     'No': 0,
#     'I stopped in the recent past': 0,
#     'I never did this': 0
# }

# for col in ['cigarettes', 'alcohol', 'marijuana']:
#     data[col] = data[col].map(substance_map)


Binary encoding complete.


In [ ]:
# Label encoding for nominal (unordered) categorical columns
# These have no inherent order, so we use integer codes assigned alphabetically

# from sklearn.preprocessing import LabelEncoder

# nominal_cols = [
#     'Shipping Address State',
#     'Title',
#     'ASIN/ISBN (Product Code)',
#     'Category',
#     'gender',
#     'sexual-orientation',
#     'state',
#     'race',
#     'life-changes'
# ]

# le = LabelEncoder()
# label_encoders = {}  # store encoders in case we need to inverse_transform later

# for col in nominal_cols:
#     data[col] = le.fit_transform(data[col].astype(str))
#     label_encoders[col] = le
#     print(f"Encoded '{col}' — {data[col].nunique()} unique values")

Encoded 'Shipping Address State' — 52 unique values
Encoded 'Title' — 505173 unique values
Encoded 'ASIN/ISBN (Product Code)' — 545395 unique values
Encoded 'Category' — 1797 unique values
Encoded 'gender' — 4 unique values
Encoded 'sexual-orientation' — 3 unique values
Encoded 'state' — 52 unique values
Encoded 'race' — 6 unique values
Encoded 'life-changes' — 6 unique values


In [ ]:
# Final check — confirm all columns are numeric

data.info()
data.describe()


<class 'pandas.DataFrame'>
Index: 1095416 entries, 0 to 1048574
Data columns (total 26 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   Purchase Price Per Unit   1095416 non-null  float64
 1   Quantity                  1095416 non-null  float64
 2   Shipping Address State    1095416 non-null  str    
 3   Title                     1095416 non-null  str    
 4   ASIN/ISBN (Product Code)  1095416 non-null  str    
 5   Category                  1095416 non-null  str    
 6   age                       1095416 non-null  int64  
 7   hispanic                  1095416 non-null  int64  
 8   race                      1095416 non-null  str    
 9   education                 1095416 non-null  int64  
 10  income                    1095416 non-null  int64  
 11  gender                    1095416 non-null  str    
 12  sexual-orientation        1095416 non-null  str    
 13  state                     1095416 non-null 

,Purchase Price Per Unit,Quantity,age,hispanic,education,income,howmany,hh-size,how-oft,order_month,order_day,order_year
count,1.095416e+06,1.095416e+06,1.095416e+06,1.095416e+06,1.095416e+06,1.095416e+06,1.095416e+06,1.095416e+06,1.095416e+06,1.095416e+06,1.095416e+06,1.095416e+06
mean,2.250059e+01,1.087660e+00,2.860737e+00,9.293912e-02,2.831238e+00,3.577873e+00,1.704967e+00,2.588987e+00,1.847878e+00,6.722656e+00,1.561563e+01,2.020468e+03
std,4.521578e+01,5.958220e-01,1.201843e+00,2.903472e-01,7.718705e-01,1.587414e+00,8.998346e-01,1.095465e+00,7.359727e-01,3.526217e+00,8.729758e+00,1.351801e+00
min,1.000000e-02,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,2.018000e+03
25%,8.490000e+00,1.000000e+00,2.000000e+00,0.000000e+00,2.000000e+00,2.000000e+00,1.000000e+00,2.000000e+00,1.000000e+00,4.000000e+00,8.000000e+00,2.020000e+03
50%,1.399000e+01,1.000000e+00,3.000000e+00,0.000000e+00,3.000000e+00,4.000000e+00,1.000000e+00,2.000000e+00,2.000000e+00,7.000000e+00,1.500000e+01,2.021000e+03
75%,2.299000e+01,1.000000e+00,4.000000e+00,0.000000e+00,3.000000e+00,5.000000e+00,2.000000e+00,4.000000e+00,2.000000e+00,1.000000e+01,2.300000e+01,2.022000e+03
max,5.949000e+03,2.190000e+02,6.000000e+00,1.000000e+00,4.000000e+00,6.000000e+00,4.000000e+00,4.000000e+00,3.000000e+00,1.200000e+01,3.100000e+01,2.023000e+03


In [ ]:
# Uncomment if approved.
# Save the fully cleaned and encoded dataset for use in downstream notebooks
# Using pickle to preserve dtypes; also saving a CSV as a backup


#data.to_pickle('/workspaces/group-project-bas-team/data/cleaned_data.pkl')
#data.to_csv('/workspaces/group-project-bas-team/data/cleaned_data.csv', index=False)

# We can't save this to our github repository because it will be over 100mb

#print(f"Saved cleaned dataset: {data.shape[0]:,} rows × {data.shape[1]} columns")